# 1. Context

This notebook analyzes OCR performance of Tesseract over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [2]:
import sys

notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent))

In [3]:
from src.viz.helper import display_box_plot

# 2. Extracted Results MetaData

In [4]:
writing_system_to_language = {'Devanagari': ['hindi','sanskrit','nepali','konkani', 'maithali', 'marathi'],
                              'tamil': ['tamil'],'telugu': ['telugu'],'Kannada': ['kannada'],'Malayalam': ['malayalam'],
                              'Bengali': ['bengali', 'assamese'],'Meetei-mayek': ['manipuri'],'Gujarati': ['gujarati'],
                              'Gurmukhi': ['punjabi'],'Odia': ['oriya'],'Arabic': ['kashmiri', 'sindhi', 'urdu'],
                              'Latin': ['english'],'Ol-chiki': ['santali']}

## 2.1. Language Results Available

In [5]:
results_root = Path("../results/tesseract")

In [6]:
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [7]:
language_to_writing_system = {
    "marathi": ["Devanagari"], "hindi": ["Devanagari"], "sanskrit": ["Devanagari"],
    "tamil": ["tamil"], "telugu": ["telugu"], "kannada": ["Kannada"],"malayalam": ["Malayalam"],
    "bengali": ["Bengali"], "assamese": ["Bengali"],"manipuri": ["Meetei-mayek"],"nepali": ["Devanagari"],
    "gujarati": ["Gujarati"], "punjabi": ["Gurmukhi"], "konkani": ["Devanagari"],"oriya": ["Odia"],"kashmiri": ["Arabic"], 
    "sindhi": ["Arabic", "Devanagari"], "urdu": ["Arabic"],"english": ["Latin"], "santali": ["Ol-chiki"],
    "maithali": ["Devanagari"]
    }

In [8]:
writing_sys_tessract_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_tessract_dict[script].append(language_res)

In [9]:
script_language_result = pd.Series(writing_sys_tessract_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [10]:
def get_language_results_path(script: str, script_language_result: pd.DataFrame) -> list[Path]:
    """Get list of results path for given script"""

    results_script_lang = script_language_result.loc[script].to_list()[0]
    results_script_lang_path = [results_root.joinpath(lang).joinpath('results.csv') for lang in results_script_lang]

    return results_script_lang_path

In [11]:
def get_script_results(script: str, script_lang_df: pd.DataFrame):
    """Get results for a given script. output contains results for languages in the script"""

    results_script_path = get_language_results_path(script, script_lang_df)

    results_list = []
    for result_path in results_script_path:
        lang = result_path.parent.name
        df_res = pd.read_csv(result_path)
        df_res['language'] = lang
        results_list.append(df_res)

    results_script = pd.concat(results_list)
    results_script['script'] = script
    return results_script

# 3. Getting All the Results

In [12]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

In [16]:
agg_results = (consolidated_df.groupby(['script', 'language']).agg(CER_AVG_L0=('cer_l0', 'median'),
                                                    CER_AVG_L1=('cer_l1', 'median'),
                                                    CER_AVG_L2=('cer_l2', 'median'),
                                                    CER_AVG_L3=('cer_l3', 'median'),
                                                    WER_AVG_L0=('wer_l0', 'median'),
                                                    WER_AVG_L1=('wer_l1', 'median'),
                                                    WER_AVG_L2=('wer_l2', 'median'),
                                                    WER_AVG_L3=('wer_l3', 'median')
                                                    ).round(3)).reset_index()
agg_results.columns = agg_results.columns.str.upper()

In [18]:
agg_results.to_clipboard(index=False) 

In [19]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

In [20]:
consolidated_df.round(3).to_clipboard(index=False)

# 4. Aggregated Results

In [ ]:
agg_results.reset_index(inplace=True)

In [ ]:
print(agg_results.to_markdown(index=False))

# 4. Results Across Various Writing System (samples)

## 4.1. Devanagari

In [ ]:
results_devanagari = get_script_results(script='Devanagari', script_lang_df=script_language_result)

### 4.1.1. Box Plot Viz

In [ ]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='CER').show()

In [ ]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='WER').show()

## 5.1. Bengali

In [ ]:
script = 'Bengali'
results_bengali = get_script_results(script='Bengali', script_lang_df=script_language_result)

### 5.1.1. Box Plot Viz

In [ ]:
display_box_plot(results_df=results_bengali, script=script, metric_type='CER').show()

# 6. Addendum

## 6.1. Assessing High WER in Hindi

In [ ]:
import jiwer

In [ ]:
script = 'Devanagari'
results_deva = get_script_results(script=script, script_lang_df=script_language_result)

In [ ]:
results_deva_hn = results_deva.loc[results_deva['language'] == 'hindi']

In [ ]:
idx = 1
gt = results_deva_hn.loc[idx]['ground_truth']
ocred = results_deva_hn.loc[idx]['ocr_output_L_0']

In [ ]:
output_wer = jiwer.process_words(gt, ocred)

In [ ]:
print(jiwer.visualize_alignment(output_wer, line_width=10))

In [ ]:
print(jiwer.visualize_error_counts(output_wer))

In [ ]:
print(gt)

In [ ]:
print(ocred)